## Step 0: Import libraries

In [ ]:
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

## Step 1: Create a synthetic dataset (with missing values)

In [ ]:
rng = np.random.default_rng(42)
n = 40
age = rng.integers(22, 60, n).astype(float)
salary = np.round(25000 + age * 900 + rng.normal(0, 6000, n), -2)
df = pd.DataFrame({
    "Country": rng.choice(["France", "Spain", "Germany"], n),
    "Age": age,
    "Salary": salary,
    "Purchased": np.where(salary + rng.normal(0, 8000, n) > 55000, "Yes", "No"),
})
df.loc[rng.choice(n, 4, replace=False), "Age"] = np.nan
df.loc[rng.choice(n, 4, replace=False), "Salary"] = np.nan
df.head(10)

## Step 2: Check missing values

In [ ]:
df.isna().sum()

## Step 3: `SimpleImputer` – mean strategy for numerical columns

In [ ]:
imputer = SimpleImputer(missing_values=np.nan, strategy="mean")
df[["Age", "Salary"]] = imputer.fit_transform(df[["Age", "Salary"]])

print("Learned means (Age, Salary):", imputer.statistics_.round(2))
print("\nMissing values after imputation:")
print(df.isna().sum())

## Step 4: `LabelEncoder` – text labels → ordinal integers

In [ ]:
le_target = LabelEncoder()
df["Purchased_Encoded"] = le_target.fit_transform(df["Purchased"])
print("Purchased mapping:", dict(zip(le_target.classes_, le_target.transform(le_target.classes_))))

df["Country_Label"] = LabelEncoder().fit_transform(df["Country"])
print("\nNote: Country_Label implies an artificial order (France < Germany < Spain),")
print("so LabelEncoder is unsuitable for nominal features in linear models.")
df[["Country", "Country_Label", "Purchased", "Purchased_Encoded"]].head()

## Step 5: One-Hot Encoding – nominal variable → binary dummy columns

In [ ]:
ohe = OneHotEncoder(sparse_output=False, dtype=int)
encoded = ohe.fit_transform(df[["Country"]])
ohe_df = pd.DataFrame(encoded, columns=ohe.get_feature_names_out(["Country"]), index=df.index)

final_df = pd.concat([df.drop(columns=["Country", "Country_Label"]), ohe_df], axis=1)
final_df.head(10)

## Step 6 (alternative): the same result with `pd.get_dummies()`

In [ ]:
pd.get_dummies(df["Country"], prefix="Country", dtype=int).head()